# Historical experiment

Preserved for provenance only. This notebook has known methodological or portability issues; use the corrected notebook linked from the root README. Outputs have been cleared.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch 
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import sklearn as sk 
from keras.models import Model, Sequential
from IPython import display
from sklearn.model_selection import train_test_split
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
books = pd.read_csv("/kaggle/input/book-recommendation-dataset/Books.csv")
ratings = pd.read_csv("/kaggle/input/book-recommendation-dataset/Ratings.csv")
users = pd.read_csv("/kaggle/input/book-recommendation-dataset/Users.csv")

In [ ]:
books.head()

In [ ]:
users.head()

In [ ]:
ratings.head()

In [ ]:
ratings_portion = ratings.head(150000)
ratings_portion.head()

In [ ]:
users_old_Id = set(ratings_portion["User-ID"])
books_ISBN = set(ratings_portion["ISBN"])

u = [i for i in range(1,len(users_old_Id)+1)]
b = [i for i in range(1,len(books_ISBN)+1)]

users_new_id = dict(zip(users_old_Id,u))
books_new_id = dict(zip(books_ISBN,b))

In [ ]:
ratings_list = ratings.values.tolist()
ratings_portion_list = ratings_portion.values.tolist()

In [ ]:
for i in ratings_list :
    if i[0] in users_new_id and i[1] in books_new_id :
        ratings_portion_list.append(i)

In [ ]:
print(len(users_new_id))
print(len(books_new_id))
print(len(ratings_portion_list))

In [ ]:
for i in range(len(ratings_portion_list)):
    ratings_portion_list[i][0] = users_new_id[ratings_portion_list[i][0]] 
    ratings_portion_list[i][1] = books_new_id[ratings_portion_list[i][1]] 

In [ ]:
ratings_portion_df = pd.DataFrame(ratings_portion_list)
ratings_portion_df.head()

In [ ]:
train, test = train_test_split(ratings_portion_df, test_size=0.1)

In [ ]:
user_input_tensor = keras.layers.Input(shape=[1],name='user-input')
user_embedding = keras.layers.Embedding(len(users_new_id)+1, 10, 
                    name='user-embedding')(user_input_tensor)
user_vector = keras.layers.Flatten(name='user-flatten')(user_embedding) 

In [ ]:
book_input_tensor = keras.layers.Input(shape=[1],name='book-input')
book_embedding = keras.layers.Embedding(len(books_new_id)+1, 10, 
                    name='book-embedding')(book_input_tensor)
book_vector = keras.layers.Flatten(name='book-flatten')(book_embedding) 

In [ ]:
print(book_vector.shape)
print(user_vector.shape)

In [ ]:
d = keras.layers.Dot(1,10)([book_vector, user_vector])
dense_1 = keras.layers.Dense(64, name='dense_1', activation='relu')(d)
dense_2= keras.layers.Dense(32, name= 'dense_2', activation='relu')(dense_1)
dense_3 = keras.layers.Dense(16, name='dense_3', activation='relu')(dense_2)
output_layer = keras.layers.Dense(1,name='output_layer', activation='relu')(dense_3)
My_model = Model([user_input_tensor, book_input_tensor], output_layer)

In [ ]:
My_model.compile(loss='mean_absolute_error', optimizer='adam')

In [ ]:
display.SVG(tf.keras.utils.model_to_dot(My_model, show_shapes=True, 
                                dpi=61).create(prog='dot', format='svg'))

In [ ]:
My_model.summary()

In [ ]:
tf.config.run_functions_eagerly(False)
fitModel = My_model.fit([train[0], train[1]], train[2], epochs=15)

In [ ]:
pd.Series(fitModel.history['loss']).plot(logy=True)
plt.xlabel("Epochs")
plt.ylabel("Train Error")
plt.show()

In [ ]:
predicted = My_model.predict([test[0], test[1]])
sk.metrics.mean_absolute_error(test[2], predicted)

In [ ]:
"""
earlystopping = keras.callbacks.EarlyStopping(monitor="val_loss",mode ="min",patience = 10)
temp_model.compile(loss='mean_absolute_error', optimizer='adam', 
                   metrics=['mean_absolute_error'])
fitModel = temp_model.fit(X_train, y_train, epochs=95, batch_size=450, 
                validation_data=(X_val, y_val), verbose=1, callbacks =[earlystopping])
results = temp_model.evaluate(X_test, y_test)"""